## TODO
* Дискрименатор
* Автоэнкодер
* Cross-domain
* Beam search with lp and cp
* SRU
* Визуализация Attn
* replace_unk по attention'у http://opennmt.net/OpenNMT/translation/unknowns/

## Tutorials
* http://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html
* https://github.com/spro/practical-pytorch/blob/master/seq2seq-translation/seq2seq-translation-batched.ipynb

## Articles
* Teaching neural networks to point to improve language modeling and translation: https://einstein.ai/research/teaching-neural-networks-to-point-to-improve-language-modeling-and-translation
* Training RNNs as Fast as CNNs : https://arxiv.org/abs/1709.02755
* Beam Search Strategies for Neural Machine Translation: https://arxiv.org/abs/1702.01806
* Unsupervised Machine Translation Using Monolingual Corpora Only: https://arxiv.org/abs/1711.00043
* Unsupervised Neural Machine Translation: https://arxiv.org/abs/1710.11041

In [1]:
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F
from torch import optim
from collections import Counter, namedtuple
import pickle
import os
import re
import random
import time
import numpy as np
from typing import List, Tuple
from torch.nn.utils.rnn import pack_padded_sequence as pack
from torch.nn.utils.rnn import pad_packed_sequence as unpack

use_cuda = torch.cuda.is_available()
print(use_cuda)

True


In [2]:
from contextlib import contextmanager
from os.path import getsize, basename
from tqdm import tqdm


@contextmanager
def tqdm_open(filename, encoding='utf8'):
    """
    Открытие файла, обёрнутое в tqdm
    """
    total = getsize(filename)

    def wrapped_line_iterator(fd):
        with tqdm(total=total, unit="B", unit_scale=True, desc=basename(filename), miniters=1) as pb:
            processed_bytes = 0
            for line in fd:
                processed_bytes += len(line)
                if processed_bytes >= 1024 * 1024:
                    pb.update(processed_bytes)
                    processed_bytes = 0
                yield line
            pb.update(processed_bytes)

    with open(filename, encoding=encoding) as fd:
        yield wrapped_line_iterator(fd)

In [3]:
class Vocabulary:
    def __init__(self, language):
        self.language = language
        self.word2index = {}
        self.word2count = Counter()
        self.index2word = ["PAD", "SOS", "EOS", "UKN"]
        if os.path.exists(self.language+".pickle"):
            self.load()

    def get_pad(self):
        return self.index2word.index("PAD")

    def get_sos(self):
        return self.index2word.index("SOS")

    def get_eos(self):
        return self.index2word.index("EOS")

    def get_ukn(self):
        return self.index2word.index("UKN")

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            if word == '':
                continue
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = len(self.index2word)
            self.word2count[word] += 1
            self.index2word.append(word)
        else:
            self.word2count[word] += 1

    def get_index(self, word):
        if word in self.word2index:
            return self.word2index[word]
        else:
            return self.get_ukn()

    def size(self):
        return len(self.index2word)

    def is_empty(self):
        return self.size() <= 4

    def shrink(self, n):
        best_words = self.word2count.most_common(n)
        self.index2word = ["PAD", "SOS", "EOS", "UKN"]
        self.word2index = {}
        self.word2count = Counter()
        for word, count in best_words:
            self.add_word(word)
            self.word2count[word] = count

    def save(self) -> None:
        with open(self.language+".pickle", "wb") as f:
            pickle.dump(self, f, pickle.HIGHEST_PROTOCOL)

    def load(self):
        with open(self.language+".pickle", "rb") as f:
            vocab = pickle.load(f)
            self.__dict__.update(vocab.__dict__)

In [4]:
Batch = namedtuple('Batch', 'input_variable, output_variable, input_lengths, output_lengths')

In [5]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_size, n_layers=3, dropout=0.1):
        super(EncoderRNN, self).__init__()
        
        num_directions = 2
        assert hidden_size % num_directions == 0
        hidden_size = hidden_size // num_directions
        
        self.embedding_dim = embedding_dim
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        self.dropout = dropout
       
        self.embedding = nn.Embedding(input_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_size, n_layers, dropout=dropout, bidirectional=True)
        
    def forward(self, input_seqs, input_lengths, hidden=None):
        embedded = self.embedding(input_seqs)
        packed = pack(embedded, input_lengths)
        outputs, hidden = self.gru(packed, hidden)
        outputs, output_lengths = unpack(outputs)
        return outputs, hidden

In [6]:
class DecoderRNN(nn.Module):
    def __init__(self, embedding_dim, hidden_size, output_size, n_layers=3, dropout=0.1):
        super(DecoderRNN, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.dropout = dropout
        
        self.embedding = nn.Embedding(output_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_size, n_layers, dropout=dropout, bidirectional=False)
        self.out = nn.Linear(hidden_size, output_size)
        
    def forward(self, input_seq, hidden):
        embedded = self.embedding(input_seq).unsqueeze(0) # S = 1 x B x N
        output, hidden = self.gru(embedded, hidden)
        output = output.squeeze(0) # S = B x N
        output = F.log_softmax(self.out(output), dim=1)
        return output, hidden

In [7]:
class Attn(nn.Module):
    def __init__(self, hidden_size):
        super(Attn, self).__init__()
        
        self.hidden_size = hidden_size
        self.attn = nn.Linear(hidden_size, hidden_size)

    def forward(self, hidden, encoder_outputs):
        max_len = encoder_outputs.size(0)
        batch_size = encoder_outputs.size(1)
        
        attn_energies = Variable(torch.zeros(batch_size, max_len))
        attn_energies = attn_energies.cuda() if use_cuda else attn_energies
        
        hidden = hidden.transpose(0, 1)
        energy = self.attn(encoder_outputs).view(batch_size, self.hidden_size, max_len)
        attn_energies = hidden.bmm(energy).transpose(0, 1).squeeze(0)  # S = B x L
        
        return F.softmax(attn_energies, dim=1).unsqueeze(1)

In [8]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, embedding_dim, hidden_size, output_size, n_layers=3, dropout=0.1, max_length=50):
        super(AttnDecoderRNN, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.dropout = dropout
        self.max_length = max_length
        
        self.embedding = nn.Embedding(output_size, embedding_dim)
        self.attn = Attn(hidden_size)
        self.gru = nn.GRU(hidden_size+embedding_dim, hidden_size, n_layers, dropout=dropout)
        self.out = nn.Linear(hidden_size, output_size)
#         self.out = nn.Linear(hidden_size*2, output_size)
    
    def forward(self, input_seq, hidden, encoder_outputs):
        # hidden: S = n_layers x B x N
        # encoder_outputs: S = L x B x N 
        embedded = self.embedding(input_seq).unsqueeze(0) # S = 1 x B x N
        
        # Calculate attention weights and apply to encoder outputs
        attn_weights = self.attn(hidden[-1].unsqueeze(0), encoder_outputs) # S = B x 1 x L
        context = attn_weights.bmm(encoder_outputs.transpose(0, 1)) # S = B x 1 X N
        context = context.transpose(0, 1) # 1 x B x N
        
        # Combine embedded input word and attended context, run through RNN
        rnn_input = torch.cat((embedded, context), 2)
        output, hidden = self.gru(rnn_input, hidden)
        
        # Final output layer
        output = output.squeeze(0) # S = B x N
        context = context.squeeze(0) # S = B x N
        output = F.log_softmax(self.out(output), dim=1)
#         output = F.log_softmax(self.out(torch.cat((output, context), 1)), dim=1)
        
        # Return final output, hidden state, and attention weights (for visualization)
        return output, hidden, attn_weights

In [9]:
class Seq2SeqAttn(nn.Module):
    def __init__(self, input_embedding_dim, output_embedding_dim, input_size, output_size, hidden_size, 
                 encoder_n_layers=3, decoder_n_layers=3, dropout=0.1, max_length=50):
        super(Seq2SeqAttn, self).__init__()
        
        self.input_embedding_dim = input_embedding_dim
        self.output_embedding_dim = output_embedding_dim
        self.input_size = input_size
        self.output_size = output_size
        self.hidden_size = hidden_size
        self.encoder_n_layers = encoder_n_layers
        self.decoder_n_layers = decoder_n_layers
        self.dropout = dropout
        self.max_length = max_length
        
        self.encoder = EncoderRNN(input_size, input_embedding_dim, hidden_size, dropout=dropout, 
                                  n_layers=encoder_n_layers)
        self.decoder = AttnDecoderRNN(output_embedding_dim, hidden_size, output_size, dropout=dropout, 
                                      max_length=max_length, n_layers=decoder_n_layers)
    
    def forward(self, batch: Batch, batch_size, criterion):
        input_variable = batch.input_variable
        target_variable = batch.output_variable
        if use_cuda:
            input_variable = input_variable.cuda()
            target_variable = target_variable.cuda()
        input_lengths = batch.input_lengths
        target_lengths = batch.output_lengths

        encoder_output, encoder_hidden = self.encoder(input_variable, input_lengths, None)
        n = encoder_hidden.size(0)
        encoder_hidden = torch.cat([encoder_hidden[0:n:2], encoder_hidden[1:n:2]], 2)

        decoder_input = Variable(torch.LongTensor([output_vocabulary.get_sos() for _ in range(batch_size)]))
        decoder_input = decoder_input.cuda() if use_cuda else decoder_input
        decoder_hidden = encoder_hidden[:self.decoder_n_layers]

        max_target_length = max(target_lengths)
        loss = 0
        for t in range(max_target_length):
            decoder_output, decoder_hidden, decoder_attn = self.decoder(decoder_input, decoder_hidden, encoder_output)
            loss += criterion(decoder_output, target_variable[t])
            decoder_input = target_variable[t]
        return loss

In [10]:
lang1 = 'en'
lang2 = 'de'
input_embedding_dim = 300
output_embedding_dim = 300
hidden_size = 500
encoder_n_layers = 2
decoder_n_layers = 2
encoder_dropout = 0.2
decoder_dropout = 0.2
max_length = 50
input_vocabulary, output_vocabulary = Vocabulary(lang1), Vocabulary(lang2)
DATASET = "/media/yallen/My Passport/Projects/UNMT/train"
INPUT_FILENAME_RAW = "/media/yallen/My Passport/Projects/UNMT/train.en"
INPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/train-sorted.en"
OUTPUT_FILENAME_RAW = "/media/yallen/My Passport/Projects/UNMT/train.de"
OUTPUT_FILENAME = "/media/yallen/My Passport/Projects/UNMT/train-sorted.de"
pairs = [(INPUT_FILENAME, OUTPUT_FILENAME)]

In [11]:
# def sort(src_filename, src_filename_sorted, tgt_filename, tgt_filename_sorted):
#     with open(src_filename, "r", encoding="utf-8") as r1, open(tgt_filename, "r", encoding="utf-8") as r2:
#         lines = list(zip(r1.readlines(), r2.readlines()))
#         lines = sorted(lines, key=lambda x: len(x[0]))
#         print(lines[10], lines[-10])
#     with open(src_filename_sorted, "w", encoding="utf-8") as w1, open(tgt_filename_sorted, "w", encoding="utf-8") as w2:
#         for line in lines:
#             w1.write(line[0])
#             w2.write(line[1])
# sort(INPUT_FILENAME_RAW, INPUT_FILENAME, OUTPUT_FILENAME_RAW, OUTPUT_FILENAME)

In [12]:
# from torchtext import data, datasets
# def tokenizer(sentence):
#         return sentence.split()
    
# src = data.Field(tokenize=tokenizer)
# trg = data.Field(tokenize=tokenizer)
# mt_train = datasets.TranslationDataset(path=DATASET, exts=('.en', '.de'), fields=(src, trg))
# src.build_vocab(mt_train, max_size=80000)
# trg.build_vocab(mt_train, max_size=40000)
# train_iter = data.BucketIterator(dataset=mt_train, batch_size=32, sort_key=lambda x: data.interleave_keys(len(x.src), len(x.trg)))

In [13]:
# next(iter(train_iter))

In [14]:
def prepare(pair_filenames: List[Tuple[str, str]], lang1: str, lang2: str):
    if input_vocabulary.is_empty() or output_vocabulary.is_empty():
        for lang1_filename, lang2_filename in pair_filenames:
            with tqdm_open(lang1_filename, encoding="utf-8") as r1, open(lang2_filename, "r", encoding="utf-8") as r2:
                for input_sentence, output_sentence in zip(r1, r2):
                    input_vocabulary.add_sentence(input_sentence.strip())
                    output_vocabulary.add_sentence(output_sentence.strip())
        input_vocabulary.shrink(100000)
        output_vocabulary.shrink(100000)
        print(input_vocabulary.word2count.most_common(50))
        print(output_vocabulary.word2count.most_common(50))
        input_vocabulary.save()
        output_vocabulary.save()
prepare(pairs, "en", "de")
# encoder = EncoderRNN(input_vocabulary.size(), input_embedding_dim, encoder_hidden_size, dropout=encoder_dropout)
# decoder = DecoderRNN(output_embedding_dim, decoder_hidden_size, output_vocabulary.size(), dropout=decoder_dropout)
# decoder = BahdanauAttnDecoderRNN(output_embedding_dim, decoder_hidden_size, output_vocabulary.size(), 
#                                  dropout=decoder_dropout,max_length=max_length)
# if use_cuda:
#     encoder = encoder.cuda()
#     decoder = decoder.cuda()

In [15]:
def indices_from_sentence(sentence: str, vocabulary: Vocabulary):
    return [vocabulary.get_index(word) for word in sentence.split(' ')] + [vocabulary.get_eos()]


def pad_seq(seq: List[int], vocabulary: Vocabulary, max_length: int):
    seq += [vocabulary.get_pad() for _ in range(max_length - len(seq))]
    return seq

class BatchGenerator:
    def __init__(self, pair_filenames: List[Tuple[str, str]], batch_size: int, max_sentence_len: int,
                 input_vocabulary: Vocabulary, output_vocabulary: Vocabulary, use_cuda: bool=True):
        self.pair_filenames = pair_filenames  # type: List[Tuple[str, str]]
        self.batch_size = batch_size  # type: int
        self.max_sentence_len = max_sentence_len  # type: int
        self.input_vocabulary = input_vocabulary
        self.output_vocabulary = output_vocabulary
        self.use_cuda = use_cuda

    def __iter__(self):
        for lang1_filename, lang2_filename in self.pair_filenames:
            input_seqs = []
            output_seqs = []
            with tqdm_open(lang1_filename, encoding='utf-8') as r1, open(lang2_filename, "r", encoding="utf-8") as r2:
                for input_sentence, output_sentence in zip(r1, r2):
                    input_sentence = input_sentence.strip()
                    output_sentence = output_sentence.strip()

                    input_sentence = indices_from_sentence(input_sentence, self.input_vocabulary)
                    output_sentence = indices_from_sentence(output_sentence, self.output_vocabulary)
                    
                    if len(input_sentence) >= self.max_sentence_len - 1 or len(output_sentence) >= self.max_sentence_len - 1:
                        continue
#                     input_sentence = input_sentence[:self.max_sentence_len - 1]
#                     output_sentence = output_sentence[:self.max_sentence_len - 1]

                    input_seqs.append(input_sentence)
                    output_seqs.append(output_sentence)
                    if len(input_seqs) == self.batch_size:
                        yield self.__process(input_seqs, output_seqs)
                        input_seqs = []
                        output_seqs = []
            if len(input_seqs) == self.batch_size:
                yield self.__process(input_seqs, output_seqs)

    def __process(self, input_seqs, output_seqs):
        input_padded, output_padded, input_lengths, output_lengths = self.__pad(input_seqs, output_seqs)
        input_variable, output_variable = self.__to_tensor(input_padded, output_padded)
        return Batch(input_variable, output_variable, input_lengths, output_lengths)

    def __pad(self, input_seqs, output_seqs):
        seq_pairs = sorted(zip(input_seqs, output_seqs), key=lambda p: len(p[0]), reverse=True)
        input_seqs, target_seqs = zip(*seq_pairs)
        input_lengths = [len(s) for s in input_seqs]
        input_padded = [pad_seq(s, self.input_vocabulary, max(input_lengths)) for s in input_seqs]
        output_lengths = [len(s) for s in target_seqs]
        output_padded = [pad_seq(s, self.output_vocabulary, max(output_lengths)) for s in target_seqs]
        return input_padded, output_padded, input_lengths, output_lengths

    def __to_tensor(self, input_padded, output_padded):
        # Turn padded arrays into (batch_size x max_len) tensors, transpose into (max_len x batch_size)
        input_variable = Variable(torch.LongTensor(input_padded), requires_grad=False).transpose(0, 1)
        output_variable = Variable(torch.LongTensor(output_padded), requires_grad=False).transpose(0, 1)
        return input_variable, output_variable

In [16]:
def train_batch(model, optimizer, criterion, batch: Batch, batch_size):
    target_lengths = batch.output_lengths
    target_tokens_count = sum(target_lengths)
    optimizer.zero_grad()
    loss = model(batch, batch_size, criterion)
    loss.backward()
    optimizer.step()
    return loss.data[0] / target_tokens_count

def train(model, pair_filenames: List[Tuple[str, str]], big_epochs: int, print_every=3000, save_every=3000, 
          learning_rate=0.001, batch_size=32):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss(ignore_index=0, size_average=False)
    batch_generator = BatchGenerator(pair_filenames, batch_size, max_length,
                                     input_vocabulary, output_vocabulary, use_cuda)
    batches = []
    for batch in batch_generator:
        batches.append(batch)
    count_batches = len(batches)
    
    print(model)
    print("Input:", batches[0].input_variable)
    print("Output:", batches[0].output_variable)
    
    for big_epoch in range(big_epochs):
        timer = time.time()
        print_loss_total = 0
        count_tokens = 0
        perm = np.random.permutation(count_batches)
        for epoch, batch_index in enumerate(perm):
            batch = batches[batch_index]
            loss = train_batch(model, optimizer, criterion, batch, batch_size)
            
            print_loss_total += loss
            count_tokens += sum(batch.input_lengths)
            if epoch % save_every == 0 and epoch != 0:
                save(model, "model.pt")
            if epoch % print_every == 0 and epoch != 0:
                print_loss_avg = print_loss_total / print_every
                print_loss_total = 0
                diff = time.time() - timer
                timer = time.time()
                src_speed = count_tokens / diff
                print('%s big epoch, %s/%s, %s src tok/s, %s sec, %.4f loss' % 
                      (big_epoch, epoch, count_batches, src_speed, diff, print_loss_avg))
                count_tokens = 0

In [17]:
def save_model(module, filename):
    state_dict = module.state_dict()
    for key in state_dict.keys():
        state_dict[key] = state_dict[key].cpu()
    torch.save({'state_dict': state_dict},filename)

def save(model, model_filename):
    save_model(model, model_filename)
    
def load(model, model_filename):
    state_dict = torch.load(model_filename)
    model.load_state_dict(state_dict['state_dict'])
    return model

In [ ]:
print("Building...")
model = Seq2SeqAttn(input_embedding_dim, output_embedding_dim, input_vocabulary.size(), output_vocabulary.size(), hidden_size)
model = model.cuda() if use_cuda else model
print("Loading...")
model = load(model, "model.pt")

Building...
Loading...


In [ ]:
print("Training...")
train(model, pairs, big_epochs=3, print_every=1000, save_every=1000, batch_size=64)

train-sorted.en:   0%|          | 0.00/455M [00:00<?, ?B/s]

Training...


train-sorted.en: 100%|█████████▉| 454M/455M [02:13<00:00, 3.40MB/s] 


Seq2SeqAttn(
  (encoder): EncoderRNN(
    (embedding): Embedding(100004, 300)
    (gru): GRU(300, 250, num_layers=3, dropout=0.1, bidirectional=True)
  )
  (decoder): AttnDecoderRNN(
    (embedding): Embedding(100004, 300)
    (attn): Attn(
      (attn): Linear(in_features=500, out_features=500)
    )
    (gru): GRU(800, 500, num_layers=3, dropout=0.1)
    (out): Linear(in_features=500, out_features=100004)
  )
)
Input: Variable containing:

Columns 0 to 12 
   16   934    25    10     4   934   934    25    25     4   540     4     4
    2     2     2     2     2     2     2     2     2     2     2     2     2

Columns 13 to 25 
   25     4     4  5037    25   934   934    25    25     4   934   934   934
    2     2     2     2     2     2     2     2     2     2     2     2     2

Columns 26 to 38 
   25     4    25   934     4   934  1966     4    14   934    25     4   540
    2     2     2     2     2     2     2     2     2     2     2     2     2

Columns 39 to 51 
    4   934 

In [22]:
def translate(model, sentence):
    indices = [indices_from_sentence(sentence, input_vocabulary)]
    input_variable = Variable(torch.LongTensor(indices)).transpose(0, 1)
    input_variable = input_variable.cuda() if use_cuda else input_variable
    input_lengths = [len(indices)]
    
    batch_size = 1 
    encoder_output, encoder_hidden = model.encoder(input_variable, input_lengths, None)
    n = encoder_hidden.size(0)
    encoder_hidden = torch.cat([encoder_hidden[0:n:2], encoder_hidden[1:n:2]], 2)

    decoder_input = Variable(torch.LongTensor([output_vocabulary.get_sos() for _ in range(batch_size)]))
    decoder_input = decoder_input.cuda() if use_cuda else decoder_input
    decoder_hidden = encoder_hidden[:model.decoder_n_layers]

    decoded_words = []
    di = 0
    while di < max_length:
        decoder_output, decoder_hidden, decoder_attn = model.decoder(decoder_input, decoder_hidden, encoder_output)
        topv, topi = decoder_output.data[0].topk(1)
        ni = topi[0]
        if ni == output_vocabulary.get_eos() or ni == output_vocabulary.get_pad():
            decoded_words.append('<EOS>')
            break
        else:
            word = output_vocabulary.index2word[ni]
            if word == "UKN" and di < len(sentence.split()):
                word = sentence.split()[di]
            decoded_words.append(word)

        decoder_input = Variable(torch.LongTensor([ni]))
        decoder_input = decoder_input.cuda() if use_cuda else decoder_input
        di += 1

    return decoded_words

In [26]:
translate(model, "Some trash .")

['-', 'Nein', '.', '<EOS>']

In [24]:
VAL_FILENAME = "src.txt"
OUTPUT_FILENAME = "pred.txt"
with open(VAL_FILENAME, "r", encoding='utf-8') as r:
    with open(OUTPUT_FILENAME, "w", encoding='utf-8') as w:
        for line in r:
            line = line.strip()
            translation = translate(line)
            w.write(" ".join(translation[:-1]) + "\n")

TypeError: translate() missing 1 required positional argument: 'sentence'

In [25]:
!perl multi-bleu.perl ref.txt < pred.txt

Use of uninitialized value $length_reference in numeric eq (==) at multi-bleu.perl line 148.
BLEU = 0, 0/0/0/0 (BP=0, ratio=0, hyp_len=0, ref_len=0)
